In [ ]:

#        Single-Shot Lensless Imaging with Physics-Guided Genetic Programming

#
#                    Code written by Ganesh M. Balasubramaniam
#
#                              Associated article:
#
#     Title: Single-Shot Lensless Imaging with Physics-Guided Genetic Programming
#
#     Authors: Ganesh M. Balasubramaniam, Xiao-Liu Chu,
#              Radhika V. Nair, Matthew R. Foreman
#
#     Preprint URL: https://doi.org/10.48550/arXiv.2604.22270
#
#     Article URL will be available after the associated manuscript is accepted.
#
#     For any queries regarding the code, please contact:
#     GMB: balasubramaniam.gm@ntu.edu.sg
#     MRF: matthew.foreman@ntu.edu.sg


#      This script uses genetic programming to generate tile-dependent hyperparameters
#      for a physics-guided gradient-descent reconstruction. The final output is saved
#      as individual reconstructed tiles and stitched full-field amplitude, intensity,
#      and phase reconstructions.


# Standard and numerical libraries
import os
import time
import math
import random
import warnings
import operator
from functools import partial

import numpy as np
import torch
import torch.nn.functional as F
import imageio.v3 as imageio
from skimage.metrics import structural_similarity as ssim
from scipy.io import savemat

# Progress bar support is optional; the reconstruction does not depend on it.
try:
    from tqdm import tqdm
    _HAS_TQDM = True
except Exception:
    _HAS_TQDM = False

# Input image and output directory. The input image is expected to match GRID * CORE in each dimension.
IMG_PATH = "USAF.png"
RESULTS_DIR = "gp"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Tiling geometry used throughout training and final reconstruction.
GRID = 4
CORE = 600
OVERLAP = 128
N = GRID * CORE

# Tiles used when scoring each GP individual.
EVAL_TILE_IDXS = [
    (0, 0), (0, 1), (0, 2), (0, 3),
    (1, 0), (1, 1), (1, 2), (1, 3),
    (2, 0), (2, 1), (2, 2), (2, 3),
    (3, 0), (3, 1), (3, 2), (3, 3)
]

# Genetic-programming parameters.
POP_SIZE = 20
N_GEN = 20
CXPB = 0.70
MUTPB = 0.30
MAX_TREE_HEIGHT = 12

# Optical and sampling parameters.
WAVELENGTH = 520e-9
DX = 1.4e-6
Z_MIN_MM = 1.5
Z_MAX_MM = 3.5
Z_MIN_M = Z_MIN_MM * 1e-3
Z_MAX_M = Z_MAX_MM * 1e-3
Z_RANGE_M = Z_MAX_M - Z_MIN_M
SAMPLE_INTERVAL = 20

# Deterministic seeds are fixed to make repeated runs easier to compare.
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
np.random.seed(0)
torch.backends.cudnn.benchmark = False


# Utilities used by the angular-spectrum propagator.
def fftshift(x):
    for dim in (-2, -1):
        n = x.size(dim)
        x = torch.roll(x, shifts=(n // 2,), dims=(dim,))
    return x


def ifftshift(x):
    for dim in (-2, -1):
        n = x.size(dim)
        x = torch.roll(x, shifts=(-n // 2,), dims=(dim,))
    return x


def compute_angular_spectrum_torch(field, z_obj_cam_m, dx, wavelength, zpad):
    """Propagate a complex field with the transfer function used in the forward model."""
    m, n = field.shape[-2:]
    M = Np = zpad if zpad >= max(m, n) else max(m, n)
    fx = torch.arange(-Np / 2, Np / 2, device=device) / (dx * Np)
    fy = torch.arange(-M / 2, M / 2, device=device) / (dx * M)
    FY, FX = torch.meshgrid(fy, fx, indexing="ij")
    H = torch.exp(-1j * np.pi * wavelength * z_obj_cam_m * (FX ** 2 + FY ** 2))
    bd = torch.cat([field[0], field[-1], field[:, 0], field[:, -1]])
    ave = bd.mean().item()
    ff = torch.full((M, Np), ave, dtype=field.dtype, device=device)
    ff[:m, :n] = field
    U = fftshift(torch.fft.fft2(ff))
    u2 = torch.fft.ifft2(ifftshift(U * H))
    return u2[:m, :n]


# The object-sensor distance is optimized in an unconstrained variable and mapped to physical bounds.
def z_from_free(z_free):
    return torch.sigmoid(z_free) * Z_RANGE_M + Z_MIN_M


def free_from_z_mm(z_mm_init):
    z_mm = float(np.clip(z_mm_init, Z_MIN_MM + 1e-6, Z_MAX_MM - 1e-6))
    t = (z_mm - Z_MIN_MM) / (Z_MAX_MM - Z_MIN_MM)
    return float(np.log(t / (1.0 - t)))


def z_mm_bounded(z_free):
    return z_from_free(z_free)[0].detach().item() * 1e3


# Loss terms used by the inner gradient-descent reconstruction.
def huber_loss(pred, target, delta):
    r = pred - target
    mask = r.abs() <= delta
    return torch.where(mask, 0.5 * r * r, delta * (r.abs() - 0.5 * delta)).mean()


def tv_loss(x):
    dx = x[:, 1:] - x[:, :-1]
    dy = x[1:, :] - x[:-1, :]
    return dx.abs().mean() + dy.abs().mean()


def binary_loss(x):
    return (4 * x * (1 - x)).mean()


def contrast_loss(x):
    return -torch.std(x)


# Tile extraction keeps an overlapped padded region but returns the core position for stitching.
def tile_coords(i, j):
    y0 = i * CORE
    x0 = j * CORE
    return y0, y0 + CORE, x0, x0 + CORE


def extract_tile_with_overlap(I_full, i, j, overlap=OVERLAP):
    y0, y1, x0, x1 = tile_coords(i, j)
    yc0 = max(0, y0 - overlap)
    yc1 = min(N, y1 + overlap)
    xc0 = max(0, x0 - overlap)
    xc1 = min(N, x1 + overlap)
    tile0 = I_full[yc0:yc1, xc0:xc1]
    H0, W0 = tile0.shape
    side = max(H0, W0)
    pad_y_top = (side - H0) // 2
    pad_y_bot = side - H0 - pad_y_top
    pad_x_left = (side - W0) // 2
    pad_x_right = side - W0 - pad_x_left
    tile = np.pad(
        tile0,
        ((pad_y_top, pad_y_bot), (pad_x_left, pad_x_right)),
        mode="edge"
    )
    y_core0 = (y0 - yc0) + pad_y_top
    x_core0 = (x0 - xc0) + pad_x_left
    return tile, y_core0, x_core0


def place_core(back, core, i, j):
    y0, y1, x0, x1 = tile_coords(i, j)
    back[y0:y1, x0:x1] = core


# Final images are written as grayscale TIFFs.
def save_uint16_image(path, arr):
    imageio.imwrite(path, (np.clip(arr, 0, 1) * 65535).astype(np.uint16), compression="none")


def run_gd_tile(I_tile_np, hp, tile_tag="00", base_outdir=None, pad_factor=1, core_origin=None, save_outputs=False):
    """Run amplitude, phase, and distance optimization on one overlapped tile."""
    outdir = None
    if save_outputs:
        if base_outdir is None:
            base_outdir = os.path.join(RESULTS_DIR, "final_tiles")
        outdir = os.path.join(base_outdir, f"tile_{tile_tag}")
        os.makedirs(outdir, exist_ok=True)

    # Work with a normalized sensor intensity for stable optimization across tiles.
    I_np = (I_tile_np / max(1e-8, I_tile_np.max())).astype(np.float32)
    I_t = torch.from_numpy(I_np).to(device)
    num_px = I_np.shape[0]
    pad = pad_factor * num_px
    zpad = num_px + 2 * pad

    n_iters = int(hp["n_iters"])
    lr_amplitude = float(hp["lr_amplitude"])
    lr_z = float(hp["lr_z"])
    lambda_tv = float(hp["lambda_tv"])
    lambda_binary = float(hp["lambda_binary"])
    lambda_contrast = float(hp["lambda_contrast"])
    delta_h = float(hp["delta_huber"])
    z_init_mm = float(hp["z_init_mm"])

    # The complex object is represented by bounded amplitude and unconstrained phase.
    a0 = np.ones_like(I_np) * 0.5
    amplitude = torch.tensor(a0, dtype=torch.float32, device=device, requires_grad=True)
    phase = torch.randn_like(amplitude) * 0.1
    phase.requires_grad = True
    z_free_init = free_from_z_mm(z_init_mm)
    z_free = torch.tensor([z_free_init], dtype=torch.float32, device=device, requires_grad=True)

    opt = torch.optim.Adam(
        [
            {"params": [amplitude, phase], "lr": lr_amplitude},
            {"params": [z_free], "lr": lr_z}
        ]
    )

    loss_history = []
    mse_history = []
    ssim_history = []
    z_history = []
    iter_history = []
    prev_Ld = None

    for it in range(n_iters):
        opt.zero_grad()
        z_phys_m = z_from_free(z_free)
        z_mm_now = z_phys_m[0].detach().item() * 1e3
        if not (Z_MIN_MM - 1e-6 <= z_mm_now <= Z_MAX_MM + 1e-6):
            raise RuntimeError(
                f"[{tile_tag}] z out of bounds: {z_mm_now:.6f} mm "
                f"(should be [{Z_MIN_MM}, {Z_MAX_MM}] mm)"
            )

        amp = amplitude.clamp(0, 1)
        phs = phase
        field = amp * torch.exp(1j * phs)
        u_f = compute_angular_spectrum_torch(field, z_phys_m[0], DX, WAVELENGTH, zpad)
        I_pr = u_f.abs() ** 2
        Ld = huber_loss(I_pr, I_t, delta_h)
        Lt = lambda_tv * (tv_loss(amp) + tv_loss(phs)) if lambda_tv > 0 else 0.0
        Lb = lambda_binary * binary_loss(amp) if lambda_binary > 0 else 0.0
        Lc = lambda_contrast * contrast_loss(amp) if lambda_contrast != 0 else 0.0
        # Data consistency is combined with policy-selected regularization weights.
        total = Ld + Lt + Lb + Lc
        total.backward()
        torch.nn.utils.clip_grad_norm_([z_free], max_norm=0.25)
        opt.step()

        with torch.no_grad():
            amplitude.clamp_(0, 1)

        if it % SAMPLE_INTERVAL == 0 or it == n_iters - 1:
            with torch.no_grad():
                iter_history.append(it + 1)
                loss_history.append(Ld.item())
                mse_val = F.mse_loss(I_pr, I_t).item()
                mse_history.append(mse_val)
                I_pr_np = I_pr.detach().cpu().numpy()
                I_pr_nm = I_pr_np / max(1e-8, I_pr_np.max())
                ssim_val = ssim(I_np, I_pr_nm, data_range=1.0)
                ssim_history.append(ssim_val)
                z_mm = z_mm_bounded(z_free)
                z_history.append(z_mm)

            if it % 100 == 0:
                print(
                    f"[tile {tile_tag}] it={it:4d}  L={Ld.item():.6f}  "
                    f"MSE={mse_val:.6f}  SSIM={ssim_val:.4f}  z={z_mm:.4f}mm"
                )

            if prev_Ld is not None and Ld.item() < 0.05 * prev_Ld:
                print(
                    f"[tile {tile_tag}] Early stop at it={it}: "
                    f"{Ld.item():.3e} < 0.05 * {prev_Ld:.3e}"
                )
                break
            prev_Ld = Ld.item()

    with torch.no_grad():
        amp_final = amplitude.clamp(0, 1)
        phase_final = phase
        z_phys_m = z_from_free(z_free)[0]
        field_fin = amp_final * torch.exp(1j * phase_final)
        u_fin = compute_angular_spectrum_torch(field_fin, z_phys_m, DX, WAVELENGTH, zpad)
        I_fin = u_fin.abs().pow(2).detach().cpu().numpy()
        I_fin_nm = I_fin / max(1e-8, I_fin.max())
        amp_np = amp_final.detach().cpu().numpy()
        phase_np = phase_final.detach().cpu().numpy()

    # Only the non-overlapped core is returned for final stitching.
    H, W = amp_np.shape
    if core_origin is None:
        c0y = (H - CORE) // 2
        c0x = (W - CORE) // 2
    else:
        c0y, c0x = core_origin

    amp_core = amp_np[c0y:c0y + CORE, c0x:c0x + CORE]
    I_core = I_fin_nm[c0y:c0y + CORE, c0x:c0x + CORE]
    phase_core = phase_np[c0y:c0y + CORE, c0x:c0x + CORE]

    if save_outputs:
        save_uint16_image(os.path.join(outdir, "amplitude_core.tiff"), amp_core)
        save_uint16_image(os.path.join(outdir, "intensity_core.tiff"), I_core)
        savemat(os.path.join(outdir, "amplitude_core.mat"), {"amplitude_core": amp_core.astype(np.float32)})
        savemat(os.path.join(outdir, "intensity_core.mat"), {"intensity_core": I_core.astype(np.float32)})
        savemat(os.path.join(outdir, "phase_core.mat"), {"phase_core": phase_core.astype(np.float32)})

    hist = {
        "iters": iter_history,
        "loss": loss_history,
        "mse": mse_history,
        "ssim": ssim_history,
        "zmm": z_history
    }

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    return amp_core, I_core, phase_core, hist


warnings.filterwarnings("ignore", category=RuntimeWarning)

# DEAP provides the symbolic GP representation and NSGA-II selection.
try:
    from deap import base, creator, tools, gp
    from deap.tools.emo import assignCrowdingDist
except Exception as e:
    raise RuntimeError("Please install DEAP: pip install deap") from e

# The primitive set maps tile-level features to reconstruction hyperparameters.
pset = gp.PrimitiveSet("HPGEN", 7)


def pdiv(a, b):
    return a / (1e-8 + abs(b))


def psqrt(x):
    return math.sqrt(abs(x))


def pexp(x):
    return math.exp(max(min(x, 6.0), -6.0))


def slog1p(x):
    return math.log1p(max(x, -0.999999))


pset.addPrimitive(operator.add, 2)
pset.addPrimitive(operator.sub, 2)
pset.addPrimitive(operator.mul, 2)
pset.addPrimitive(pdiv, 2)
pset.addPrimitive(min, 2)
pset.addPrimitive(max, 2)
pset.addPrimitive(abs, 1)
pset.addPrimitive(math.tanh, 1)
pset.addPrimitive(slog1p, 1)
pset.addPrimitive(psqrt, 1)
pset.addPrimitive(pexp, 1)
pset.addEphemeralConstant("rand", partial(random.uniform, -3, 3))
pset.renameArguments(ARG0="cx")
pset.renameArguments(ARG1="cy")
pset.renameArguments(ARG2="mI")
pset.renameArguments(ARG3="sI")
pset.renameArguments(ARG4="gM")
pset.renameArguments(ARG5="gS")
pset.renameArguments(ARG6="bias")


def init_tree():
    return gp.genHalfAndHalf(pset=pset, min_=1, max_=3)


def bound_map(val, lo, hi):
    y = 1.0 / (1.0 + math.exp(-val))
    return lo + (hi - lo) * y


def compile_policy(ind):
    """Compile the eight GP trees into a callable tile-wise hyperparameter policy."""
    fns = [gp.compile(expr=t, pset=pset) for t in ind[:8]]

    def gen_hp(feat):
        raw = [fn(*feat) for fn in fns]
        n_iters        = int(round(bound_map(raw[0],  1000, 4000)))
        lr_amplitude   = float(bound_map(raw[1],  1e-3, 1e-2))
        lr_z           = float(bound_map(raw[2],  1e-4, 3e-3))
        lambda_tv      = float(bound_map(raw[3],  1e-4, 1e-2))
        lambda_binary  = float(bound_map(raw[4],  1e-5,  1e-3))
        lambda_contrast= -float(bound_map(raw[5],  1e-5,  1e-3))
        delta_huber    = float(bound_map(raw[6],  1e-4, 1e-2))
        z_init_mm      = float(bound_map(raw[7],  Z_MIN_MM, Z_MAX_MM))
        return dict(
            n_iters=n_iters,
            lr_amplitude=lr_amplitude,
            lr_z=lr_z,
            lambda_tv=lambda_tv,
            lambda_binary=lambda_binary,
            lambda_contrast=lambda_contrast,
            delta_huber=delta_huber,
            z_init_mm=z_init_mm
        )

    return gen_hp


# The fitness rewards reconstruction fidelity and penalizes runtime and tree size.
if "FitnessMulti" not in creator.__dict__:
    creator.create("FitnessMulti", base.Fitness, weights=(+1.0, -0.25, -0.02))
if "Individual" not in creator.__dict__:
    creator.create("Individual", list, fitness=creator.FitnessMulti)

toolbox = base.Toolbox()
toolbox.register("expr", init_tree)


# Each individual consists of eight trees, one for each reconstructed hyperparameter.
def init_individual():
    return creator.Individual([tools.initIterate(gp.PrimitiveTree, toolbox.expr) for _ in range(8)])


toolbox.register("individual", init_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("expr_mut", gp.genFull, min_=0, max_=2)


# Crossover and mutation operate on one randomly selected tree within the individual.
def cx_individual(ind1, ind2):
    gi = random.randrange(8)
    gp.cxOnePoint(ind1[gi], ind2[gi])
    return ind1, ind2


def mut_individual(ind):
    gi = random.randrange(8)
    (ind[gi],) = gp.mutUniform(ind[gi], expr=toolbox.expr_mut, pset=pset)
    return (ind,)


toolbox.register("mate", cx_individual)
toolbox.register("mutate", mut_individual)
toolbox.decorate(
    "mate",
    gp.staticLimit(key=lambda ind: max(t.height for t in ind), max_value=MAX_TREE_HEIGHT)
)
toolbox.decorate(
    "mutate",
    gp.staticLimit(key=lambda ind: max(t.height for t in ind), max_value=MAX_TREE_HEIGHT)
)
toolbox.register("select", tools.selNSGA2)

# Load the training measurement once and compute global intensity descriptors.
I_full = imageio.imread(IMG_PATH, mode="L").astype(np.float32) / 255.0
assert I_full.shape == (N, N), f"Image must be {N}x{N}, got {I_full.shape}"
gM, gS = float(I_full.mean()), float(I_full.std())
_IND_SEQ = 0


def tile_features(i, j, I_tile):
    """Return spatial and intensity features used as GP policy inputs."""
    y0, y1, x0, x1 = tile_coords(i, j)
    cy = (y0 + y1) / 2.0 / N
    cx = (x0 + x1) / 2.0 / N
    mI = float(I_tile.mean())
    sI = float(I_tile.std())
    return cx, cy, mI, sI, gM, gS, 1.0


def assign_id(ind):
    global _IND_SEQ
    if not hasattr(ind, "_id"):
        _IND_SEQ += 1
        ind._id = _IND_SEQ
    return ind._id


def evaluate_ind(ind):
    """Evaluate one GP individual using the mean SSIM over the selected tiles."""
    start = time.time()
    gen_hp = compile_policy(ind)
    ssim_vals = []

    for ii, jj in EVAL_TILE_IDXS:
        I_tile, y_core0, x_core0 = extract_tile_with_overlap(I_full, ii, jj, overlap=OVERLAP)
        feat = tile_features(ii, jj, I_tile)
        hp = gen_hp(feat)
        _, _, _, hist = run_gd_tile(
            I_tile,
            hp,
            tile_tag=f"{ii}{jj}",
            pad_factor=1,
            core_origin=(y_core0, x_core0),
            save_outputs=False
        )
        ssim_vals.append(hist["ssim"][-1] if len(hist["ssim"]) > 0 else 0.0)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

    elapsed = time.time() - start
    mean_ssim = float(np.mean(ssim_vals)) if len(ssim_vals) else 0.0
    size = sum(len(t) for t in ind[:8])
    return mean_ssim, elapsed, size


toolbox.register("evaluate", evaluate_ind)


def full_reconstruct(best_ind, tag="final", out_dir=None, image=None, pad_factor=1, overlap=None, save_tiles=True):
    """Apply the selected policy to all tiles and save the final stitched reconstruction."""
    global gM, gS

    if image is None:
        I_src = I_full
        gM, gS = float(I_full.mean()), float(I_full.std())
    else:
        assert image.shape == (N, N), f"Override image must be {N}x{N}, got {image.shape}"
        I_src = image
        gM, gS = float(I_src.mean()), float(I_src.std())

    if overlap is None:
        overlap_val = OVERLAP
    else:
        try:
            overlap_val = int(overlap)
        except Exception:
            overlap_val = OVERLAP
        if overlap_val < 0:
            overlap_val = 0

    gen_hp = compile_policy(best_ind)
    amp_back = np.zeros((N, N), dtype=np.float32)
    I_back = np.zeros((N, N), dtype=np.float32)
    phase_back = np.zeros((N, N), dtype=np.float32)
    hist_all = {}

    if out_dir is None:
        out_dir = os.path.join(RESULTS_DIR, tag)
    os.makedirs(out_dir, exist_ok=True)
    tile_base = os.path.join(out_dir, "tiles")

    if save_tiles:
        os.makedirs(tile_base, exist_ok=True)

    # Reconstruct each tile independently, then place the core into the full-field mosaic.
    for ii in range(GRID):
        for jj in range(GRID):
            I_tile, y_core0, x_core0 = extract_tile_with_overlap(I_src, ii, jj, overlap=overlap_val)
            feat = tile_features(ii, jj, I_tile)
            hp = gen_hp(feat)
            amp_core, I_core, phase_core, hist = run_gd_tile(
                I_tile,
                hp,
                tile_tag=f"{ii}{jj}",
                base_outdir=tile_base,
                pad_factor=pad_factor,
                core_origin=(y_core0, x_core0),
                save_outputs=save_tiles
            )
            place_core(amp_back, amp_core, ii, jj)
            place_core(I_back, I_core, ii, jj)
            place_core(phase_back, phase_core, ii, jj)
            hist_all[f"{ii}_{jj}"] = hist

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.synchronize()

    save_uint16_image(os.path.join(out_dir, f"{tag}_amplitude.tiff"), amp_back)
    save_uint16_image(os.path.join(out_dir, f"{tag}_reconstructed_intensity.tiff"), I_back)
    savemat(os.path.join(out_dir, f"{tag}_phase.mat"), {"phase": phase_back.astype(np.float32)})

    return amp_back, I_back, phase_back, hist_all


def eval_with_status(ind, gen, idx, total):
    """Evaluate an individual and attach its multi-objective fitness values."""
    iid = assign_id(ind)
    label = f"G{gen} ind {idx}/{total} [id={iid}]"
    print(f"\n▶ {label} evaluating...")
    t0 = time.perf_counter()
    ssim_val, elapsed, size = toolbox.evaluate(ind)
    wall = time.perf_counter() - t0
    ind.fitness.values = float(ssim_val), float(elapsed), int(size)
    print(
        f"✓ {label} SSIM={ssim_val:.4f} time={elapsed:.1f}s "
        f"size={size} wall={wall:.1f}s"
    )
    return ind


def run_gp():
    """Run the GP loop and return the best individual according to SSIM."""
    print("=" * 70)
    print("GP-on-GD Tiled Reconstruction (DEAP, COMPLEX OBJECT)")
    print("=" * 70)
    print(f"Image: {I_full.shape}, tiles: {GRID}x{GRID}, core={CORE}, overlap={OVERLAP}")
    print("Evaluating tiles:", EVAL_TILE_IDXS)
    print("=" * 70)

    pop = toolbox.population(n=POP_SIZE)

    for ind in pop:
        assign_id(ind)

    print("\n===== GP Generation 0 / initial evaluation =====")
    if _HAS_TQDM:
        seq = list(enumerate(pop, 1))
        for idx, ind in tqdm(seq, desc="Initial eval", ncols=90):
            eval_with_status(ind, gen=0, idx=idx, total=len(pop))
    else:
        for idx, ind in enumerate(pop, 1):
            eval_with_status(ind, gen=0, idx=idx, total=len(pop))

    pop = tools.selNSGA2(pop, k=POP_SIZE)

    for gen in range(1, N_GEN + 1):
        print(f"\n===== GP Generation {gen}/{N_GEN} =====")
        assignCrowdingDist(pop)
        k = len(pop)

        if k % 4 == 0:
            parents = tools.selTournamentDCD(pop, k)
        else:
            parents = tools.selTournamentDCD(pop, k - (k % 4))

        offspring = [toolbox.clone(ind) for ind in parents]
        ops_counts = {"cx": 0, "mut": 0, "clone": 0}

        for c1, c2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(c1, c2)
                if hasattr(c1.fitness, "values"):
                    del c1.fitness.values
                if hasattr(c2.fitness, "values"):
                    del c2.fitness.values
                ops_counts["cx"] += 2

        for m in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(m)
                if hasattr(m.fitness, "values"):
                    del m.fitness.values
                ops_counts["mut"] += 1

        for m in offspring:
            if not hasattr(m.fitness, "values"):
                ops_counts["clone"] += 1

        print(
            f"Variation: crossovered={ops_counts['cx']} "
            f"mutated={ops_counts['mut']} cloned={ops_counts['clone']}"
        )

        invalid = [ind for ind in offspring if not ind.fitness.valid]
        if invalid:
            if _HAS_TQDM:
                seq = list(enumerate(invalid, 1))
                for idx, ind in tqdm(seq, desc=f"G{gen} eval", ncols=90):
                    eval_with_status(ind, gen=gen, idx=idx, total=len(invalid))
            else:
                for idx, ind in enumerate(invalid, 1):
                    eval_with_status(ind, gen=gen, idx=idx, total=len(invalid))

        pop = toolbox.select(pop + offspring, k=POP_SIZE)
        best = max(pop, key=lambda ind: ind.fitness.values[0])
        best_id = getattr(best, "_id", None)
        print(
            f"★ Generation {gen} best "
            f"SSIM={best.fitness.values[0]:.4f} "
            f"time={best.fitness.values[1]:.1f}s "
            f"size={int(best.fitness.values[2])} id={best_id}"
        )

    best = max(pop, key=lambda ind: ind.fitness.values[0])
    return best


# Final run: evolve the policy and save the final tiles and stitched outputs.
if __name__ == "__main__":
    best_ind = run_gp()
    final_out = os.path.join(RESULTS_DIR, "final_best")
    os.makedirs(final_out, exist_ok=True)
    full_reconstruct(
        best_ind,
        tag="final",
        out_dir=final_out,
        pad_factor=1,
        save_tiles=True
    )
    print("\nDone. Final tiles and final stitched outputs are in:", final_out)
